In [ ]:
!pip install datasets
!pip install -U datasets
!pip install transformers
!pip install torch

In [ ]:
from datasets import load_dataset
ds = load_dataset("roneneldan/TinyStories")

In [ ]:
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

encoder = tiktoken.get_encoding("gpt2")

def process(example):
    ids = encoder.encode_ordinary(example['text'])
    outputs = {
        "ids": ids,
        "len": len(ids)
    }
    return outputs

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns = ["text"],
        desc = "Tokenizing the Text",
        num_proc = 8,
    )

    for split,dset in tokenized.items():
        arr_len = np.sum(dset["len"],dtype = np.unit64)
        filename = f'{split}.bin'
        dtype = np.uint16
        arr = np.memmap(filename,dtype = dtype,mode= 'w+',shape = (arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches),desc=f"Writing {filename}"):
            batch = dset.shard(num_shards =total_batches,index = batch_idx,contigious = True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            arr[idx:idx+len(arr_batch)] = arr_batch
            idx+= len(arr_batch)
        arr.flush()
        


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
def get_batch(split):
    if split == 'train':
        data = np.memmap('train.bin',dtype = np.uint16,mode = 'r')
    else:
        data = np.memmap('validation.bin',dtype = np.uint16,mode = 'r')
    ix = torch.randint(0,len(data)-block_size,(batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.uint64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    if device_type == "cuda":
        x,y = x.pin_memory().to(device,non_blocking=True) , y.pin_memory().to(device,non_blocking=True)
    else:
        x,y = x.to(device), y.to(device)
    return x,y

In [ ]:


def apply_rotary_emb(x:torch.Tensor,cos:torch.Tensor, sin:torch.Tensor)-> torch.Tensor:
    cos = cos.unsqueeze(-2).to(x.dtype)
    sin = sin.unsqueeze(-2).to(x.dtype)
    x1,x2 = torch.chunk(x,2,dim=-1)
    o1 = x1*cos - x2*sin
    o2 = x2*cos + x1*sin
    return torch.cat((o1,o2),dim=-1)


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self,num_features:int,eps:float=1e-05,device: torch.device = None):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.scale = torch.nn.Parameter(torch.ones(num_features,device,dtype= torch.float32))
    
    def forward(self,x:torch.Tensor)-> torch.Tensor:
        assert x.shape[-1]==self.num_features
        t,dtype = x.float() , x.dtype
        t = t*torch.rsqrt(torch.mean(t**2,dim=-1,keepdim= True)+self.eps)
        return (t*self.scale).to(dtype)

